In [11]:
import requests
import time

# --- CONFIGURATION ---
WORKERS = [
    "https://wincing-alumni-lark.ngrok-free.dev/generate",
    "https://refined-goldfish-kitten.ngrok-free.dev/generate"
]

class MasterLoadBalancer:
    def __init__(self, worker_urls):
        self.worker_urls = worker_urls
        self.healthy_workers = []

    def scan_health(self):
        """🩺 Scans the cluster and only keeps active workers"""
        print("🩺 SCANNING CLUSTER HEALTH...")
        active = []
        for url in self.worker_urls:
            health_url = url.replace("/generate", "/health")
            try:
                resp = requests.get(health_url, timeout=3)
                if resp.status_code == 200:
                    active.append(url)
            except:
                continue
        self.healthy_workers = active
        return len(active)

    def ask_sequential(self, prompts):
        """🚀 Processes all prompts one-by-one using healthy workers"""
        if not self.healthy_workers:
            print("🚨 CRITICAL: No healthy workers found!")
            return

        num_healthy = len(self.healthy_workers)
        print(f"✅ Found {num_healthy} healthy worker(s). Starting sequential processing...")

        overall_start = time.time()

        for i, p in enumerate(prompts):
            # Round Robin: Pick the next available healthy worker
            worker_url = self.healthy_workers[i % num_healthy]

            print(f"📢 MASTER: Sending '{p}' to {worker_url}...")
            start = time.time()
            try:
                response = requests.post(worker_url, json={"prompt": p}, timeout=120)
                end = time.time()

                if response.status_code == 200:
                    data = response.json()
                    print(f"✅ SUCCESS from {data.get('worker_id')} in {round(end-start, 2)}s")
                    print(f"💡 ANSWER: {data.get('response').strip()}\n" + "-"*30)
                else:
                    print(f"❌ Worker Error: {response.status_code}")
            except Exception as e:
                print(f"❌ Connection Error: {e}")

        overall_end = time.time()
        total_time = round(overall_end - overall_start, 2)
        print(f"\n⏱️ TOTAL SEQUENTIAL TIME: {total_time} seconds")

# --- EXECUTION ---
cluster = MasterLoadBalancer(WORKERS)

# 1. First, find out who is alive
if cluster.scan_health() > 0:
    # 2. Then, process EVERY prompt in the list
    test_prompts = ["Who is Modi?", "Who is Trump?"]
    cluster.ask_sequential(test_prompts)
else:
    print("❌ Cannot proceed: All workers are offline.")

🩺 SCANNING CLUSTER HEALTH...
✅ Found 2 healthy worker(s). Starting sequential processing...
📢 MASTER: Sending 'Who is Modi?' to https://wincing-alumni-lark.ngrok-free.dev/generate...
✅ SUCCESS from Colab-Worker-1 in 14.95s
💡 ANSWER: Modi is a former Indian prime minister who served as the 14th and current leader of the Bharatiya Janata Party (BJP) since 2014. He also served as the Chief Minister of Gujarat from 2001 to 2014. He was elected as the Prime Minister of India for the second consecutive term in 2014. Modi is widely regarded as one of the most successful and influential prime ministers in Indian history, with his policies and approach to governance being widely lauded across the nation.
------------------------------
📢 MASTER: Sending 'Who is Trump?' to https://refined-goldfish-kitten.ngrok-free.dev/generate...
✅ SUCCESS from Colab-Worker-2 in 11.04s
💡 ANSWER: Trump is the 45th president of the United States. He was elected in November 2016, defeating former Secretary of State

In [13]:
import requests
import time
from concurrent.futures import ThreadPoolExecutor

# --- YOUR TWO INDEPENDENT TUNNELS ---
WORKERS = [
    "https://wincing-alumni-lark.ngrok-free.dev/generate",
    "https://refined-goldfish-kitten.ngrok-free.dev/generate"
]

def call_worker(worker_url, prompt):
    print(f"🚀 Sending task to {worker_url}...")
    start = time.time()
    try:
        response = requests.post(worker_url, json={"prompt": prompt}, timeout=120)
        end = time.time()
        data = response.json()

        # CHANGED: Now we return a dictionary containing the actual AI answer
        return {
            "status": "success",
            "worker_id": data.get('worker_id'),
            "response": data.get('response'), # This is the AI's text
            "time": round(end-start, 2),
            "prompt": prompt
        }
    except Exception as e:
        return {"status": "error", "url": worker_url, "message": str(e)}

def is_worker_healthy(url):
    health_url = url.replace("/generate", "/health")
    try:
        resp = requests.get(health_url, timeout=3)
        return resp.status_code == 200
    except:
        return False

# --- THE EXECUTION ---
print("🩺 SCANNING CLUSTER HEALTH...")
healthy_workers = [w for w in WORKERS if is_worker_healthy(w)]

if not healthy_workers:
    print("🚨 CRITICAL: No healthy workers found! Check your Colab tabs.")
else:
    print(f"🔥 STARTING PARALLEL REQUESTS on {len(healthy_workers)} workers...")
    overall_start = time.time()

    # We use ThreadPoolExecutor to fire BOTH requests at the same time
    with ThreadPoolExecutor(max_workers=len(healthy_workers)) as executor:
        prompts = ["Tell me about trump.", "Tell me about modi."]
        results = list(executor.map(call_worker, healthy_workers, prompts))

    overall_end = time.time()

    # --- PRINTING THE ACTUAL ANSWERS ---
    print("\n" + "="*50)
    print("           PARALLEL INFERENCE RESULTS")
    print("="*50)

    for res in results:
        if res["status"] == "success":
            print(f"🤖 WORKER: {res['worker_id']} ({res['time']}s)")
            print(f"❓ PROMPT: {res['prompt']}")
            print(f"💡 ANSWER: {res['response'].strip()}") # .strip() removes extra spaces
            print("-" * 30)
        else:
            print(f"❌ ERROR on {res['url']}: {res['message']}")

    total_time = round(overall_end - overall_start, 2)
    print(f"\n⏱️ TOTAL SYSTEM TIME: {total_time} seconds")

🩺 SCANNING CLUSTER HEALTH...
🔥 STARTING PARALLEL REQUESTS on 2 workers...
🚀 Sending task to https://wincing-alumni-lark.ngrok-free.dev/generate...
🚀 Sending task to https://refined-goldfish-kitten.ngrok-free.dev/generate...

           PARALLEL INFERENCE RESULTS
🤖 WORKER: Colab-Worker-1 (21.49s)
❓ PROMPT: Tell me about trump.
💡 ANSWER: Trump is a US President who was elected in 2016. He is the 45th President of the United States of America. Before being elected, he served as the 35th Governor of New York from 2011 to 2015. Trump has been a well-known figure for his controversial statements and policies, such as his refusal to denounce white nationalism and his attacks on the media. He has also faced numerous legal challenges, including lawsuits filed against him by the House of Representatives and the Senate. His presidency has been marked by numerous scandals, such as the Trump Tower meeting and the Russian interference in the 2016 US elections. Despite these challenges, Trump remains

In [12]:
import requests
import time

# --- 1. CONFIGURATION ---
# Use the unique Ngrok URLs from your Worker 1 and Worker 2 tabs
WORKERS = [
    "https://wincing-alumni-lark.ngrok-free.dev/generate",
    "https://refined-goldfish-kitten.ngrok-free.dev/generate"
]

class SequentialMaster:
    def __init__(self, worker_urls):
        self.worker_urls = worker_urls
        self.healthy_workers = []

    def scan_health(self):
        """🩺 Scans the cluster to find active workers"""
        print("🩺 SCANNING CLUSTER HEALTH...")
        active = []
        for url in self.worker_urls:
            health_url = url.replace("/generate", "/health")
            try:
                resp = requests.get(health_url, timeout=3)
                if resp.status_code == 200:
                    active.append(url)
            except:
                continue
        self.healthy_workers = active
        return len(active)

    def run_sequential(self, prompts):
        """🚀 Processes all prompts one-by-one"""
        if not self.healthy_workers:
            print("🚨 CRITICAL: No healthy workers found!")
            return

        num_healthy = len(self.healthy_workers)
        print(f"✅ Found {num_healthy} healthy worker(s). Starting sequential processing...")

        overall_start = time.time() # Start total timer

        for i, p in enumerate(prompts):
            # Round Robin: Rotate between workers for every prompt
            worker_url = self.healthy_workers[i % num_healthy]

            print(f"📢 Request {i+1}/10: Sending to {worker_url}...")

            start = time.time()
            try:
                # Tech Stack: Synchronous POST request using 'requests'
                response = requests.post(worker_url, json={"prompt": p}, timeout=120)
                end = time.time()

                if response.status_code == 200:
                    data = response.json()
                    worker_id = data.get('worker_id')
                    answer = data.get('response').strip()
                    latency = round(end - start, 2)

                    print(f"✅ SUCCESS from {worker_id} in {latency}s")
                    print(f"💡 ANSWER: {answer}\n" + "-"*40)
                else:
                    print(f"❌ Worker Error: {response.status_code}")
            except Exception as e:
                print(f"❌ Connection Error: {e}")

        overall_end = time.time()
        total_time = round(overall_end - overall_start, 2)
        print(f"\n⏱️ TOTAL SEQUENTIAL EXECUTION TIME: {total_time} seconds")

# --- EXECUTION ---
prompts = [
    "What is the capital of France?",
    "Tell me a short joke about robots.",
    "Explain gravity in one sentence.",
    "Who wrote Romeo and Juliet?",
    "What is 2 + 2?",
    "Name a famous scientist.",
    "What is the color of the sky?",
    "Translate 'Hello' to Spanish.",
    "What is the largest ocean?",
    "How many legs does a spider have?"
]

cluster = SequentialMaster(WORKERS)

if cluster.scan_health() > 0:
    cluster.run_sequential(prompts)
else:
    print("❌ All workers are offline. Cannot proceed.")

🩺 SCANNING CLUSTER HEALTH...
✅ Found 2 healthy worker(s). Starting sequential processing...
📢 Request 1/10: Sending to https://wincing-alumni-lark.ngrok-free.dev/generate...
✅ SUCCESS from Colab-Worker-1 in 1.75s
💡 ANSWER: The capital of France is Paris.
----------------------------------------
📢 Request 2/10: Sending to https://refined-goldfish-kitten.ngrok-free.dev/generate...
✅ SUCCESS from Colab-Worker-2 in 11.75s
💡 ANSWER: Sure, here's a short joke about robotics:

Robot: "Hi, can you help me pick up my keys?"

Robot: "Sure, but can you tell me what they look like?"

Robot: "No, they're invisible to the human eye."

Robot: "Oh, that's too bad. But at least they're very smart."

Robot: "Yeah,
----------------------------------------
📢 Request 3/10: Sending to https://wincing-alumni-lark.ngrok-free.dev/generate...
✅ SUCCESS from Colab-Worker-1 in 10.18s
💡 ANSWER: Gravity is the force that holds the Earth, other planets, and other celestial bodies in orbit. It is a force that arises 

In [15]:
import requests
import time
from concurrent.futures import ThreadPoolExecutor

# --- 1. CONFIGURATION (Tech Stack: Pyngrok Tunnels) ---
# Update these URLs with the ones generated in your Worker Colab tabs
WORKERS = [
    "https://wincing-alumni-lark.ngrok-free.dev/generate",
    "https://refined-goldfish-kitten.ngrok-free.dev/generate"
]

# --- 2. UTILITY FUNCTIONS (Tech Stack: FastAPI & Requests) ---
def is_worker_healthy(url):
    """Pings the /health endpoint to verify worker availability"""
    health_url = url.replace("/generate", "/health")
    try:
        resp = requests.get(health_url, timeout=3)
        return resp.status_code == 200
    except:
        return False

def call_worker(worker_url, prompt):
    """Sends a prompt to a specific worker and measures latency"""
    start_time = time.time()
    try:
        # Tech Stack: POST request to FastAPI endpoint
        response = requests.post(worker_url, json={"prompt": prompt}, timeout=120)
        data = response.json()
        end_time = time.time()

        return {
            "status": "success",
            "worker": data.get('worker_id'),
            "response": data.get('response').strip(),
            "prompt": prompt,
            "latency": round(end_time - start_time, 2)
        }
    except Exception as e:
        return {"status": "error", "url": worker_url, "message": str(e)}

# --- 3. EXECUTION LOGIC (Tech Stack: ThreadPoolExecutor for Concurrency) ---
# 10 Basic prompts for TinyLlama testing
test_prompts = [
    "What is the capital of France?",
    "Tell me a short joke about robots.",
    "Explain gravity in one sentence.",
    "Who wrote Romeo and Juliet?",
    "What is 2 + 2?",
    "Name a famous scientist.",
    "What is the color of the sky?",
    "Translate 'Hello' to Spanish.",
    "What is the largest ocean?",
    "How many legs does a spider have?"
]

print("🩺 SCANNING CLUSTER HEALTH...")
healthy_workers = [w for w in WORKERS if is_worker_healthy(w)]

if not healthy_workers:
    print("🚨 CRITICAL: No workers online. System halted.")
else:
    print(f"✅ Found {len(healthy_workers)} healthy worker(s).")
    print(f"🔥 STARTING DISTRIBUTED INFERENCE ON {len(test_prompts)} PROMPTS...")

    overall_start = time.time()
    results = []

    # Parallel processing: Firing requests at all workers simultaneously
    with ThreadPoolExecutor(max_workers=len(healthy_workers)) as executor:
        futures = []
        for i, p in enumerate(test_prompts):
            # Round Robin logic: Distributes tasks evenly across workers
            target_worker = healthy_workers[i % len(healthy_workers)]
            futures.append(executor.submit(call_worker, target_worker, p))

        for future in futures:
            results.append(future.result())

    overall_end = time.time()
    total_time = round(overall_end - overall_start, 2)

    # --- 4. PERFORMANCE REPORT ---
    print("\n" + "="*60)
    print("           DISTRIBUTED SYSTEM PERFORMANCE REPORT")
    print("="*60)

    for res in results:
        if res["status"] == "success":
            print(f"❓ PROMPT  : {res['prompt']}")
            print(f"🤖 WORKER  : {res['worker']}")
            print(f"⏱️  LATENCY : {res['latency']}s")
            print(f"💡 ANSWER  : {res['response']}")
            print("-" * 60)
        else:
            print(f"❌ FAILED  : {res['message']}")

    print(f"🚀 TOTAL CLUSTER EXECUTION TIME: {total_time}s")
    print("="*60)

🩺 SCANNING CLUSTER HEALTH...
✅ Found 2 healthy worker(s).
🔥 STARTING DISTRIBUTED INFERENCE ON 10 PROMPTS...

           DISTRIBUTED SYSTEM PERFORMANCE REPORT
❓ PROMPT  : What is the capital of France?
🤖 WORKER  : Colab-Worker-1
⏱️  LATENCY : 1.78s
💡 ANSWER  : The capital of France is Paris.
------------------------------------------------------------
❓ PROMPT  : Tell me a short joke about robots.
🤖 WORKER  : Colab-Worker-2
⏱️  LATENCY : 12.57s
💡 ANSWER  : Sure, here's a short joke about robotics:

A robotics engineer is working on a project to create a robot that can build a house. They arrive at the construction site, only to find a group of human builders already working on the same project.

The engineer rolls up their sleeves and starts working on a prototype of the robot. After a few minutes, they stop and exclaim, "I've got an idea!"
------------------------------------------------------------
❓ PROMPT  : Explain gravity in one sentence.
🤖 WORKER  : Colab-Worker-1
⏱️  LATENCY : 6